# 05 — Complete Pipeline

## 1. Goal

Combine claim extraction, retrieval, and NLI into one pipeline.

## 2. Import Libraries

In [1]:
import spacy
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
import pandas as pd

## 3. Load Models

In [2]:
nlp = spacy.load("en_core_web_sm")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
nli = pipeline(
    "text-classification",
    model="cross-encoder/nli-MiniLM2-L6-H768"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## 4. Source Document

In [3]:
source_text = """
The James Webb Space Telescope is a space telescope designed to conduct infrared astronomy.
It was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre.
The telescope operates near the second Lagrange point, approximately 1.5 million kilometers from Earth.
"""

## 5. LLM Generated Answer

In [4]:
answer = """
The James Webb Space Telescope was launched in 2021.
It was launched using an Ariane 5 rocket.
The telescope is approximately 1.5 million kilometers from Earth.
The telescope was launched from India.
"""

## 6. Extract Claims

In [6]:
doc = nlp(answer)
claims = [sentence.text.strip() for sentence in doc.sents]
claims

['The James Webb Space Telescope was launched in 2021.',
 'It was launched using an Ariane 5 rocket.',
 'The telescope is approximately 1.5 million kilometers from Earth.',
 'The telescope was launched from India.']

## 7. Create Source Chunks

In [7]:
source_doc = nlp(source_text)
chunks = [
    sentence.text.strip()
    for sentence in source_doc.sents
]
chunks

['The James Webb Space Telescope is a space telescope designed to conduct infrared astronomy.',
 'It was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre.',
 'The telescope operates near the second Lagrange point, approximately 1.5 million kilometers from Earth.']

## 8. Retrieve Relevant Evidence

In [8]:
chunk_embeddings = embedding_model.encode(chunks)
results = []
for claim in claims:
    claim_embedding = embedding_model.encode(claim)
    similarities = cosine_similarity(
        [claim_embedding],
        chunk_embeddings
    )[0]
    best_index = similarities.argmax()
    results.append({
        "claim": claim,
        "evidence": chunks[best_index],
        "similarity": similarities[best_index]
    })
retrieval_df = pd.DataFrame(results)
retrieval_df

,claim,evidence,similarity
0,The James Webb Space Telescope was launched in...,The James Webb Space Telescope is a space tele...,0.709175
1,It was launched using an Ariane 5 rocket.,"It was launched on December 25, 2021, aboard a...",0.719603
2,The telescope is approximately 1.5 million kil...,The telescope operates near the second Lagrang...,0.809263
3,The telescope was launched from India.,The telescope operates near the second Lagrang...,0.519770


## 9. Run NLI

In [9]:
predictions = []
for _, row in retrieval_df.iterrows():
    result = nli(
        f"{row['evidence']} </s></s> {row['claim']}"
    )
    predictions.append({
        "label": result[0]["label"].upper(),
        "score": result[0]["score"]
    })

## 10. Combine Results

In [10]:
retrieval_df["nli_label"] = [
    result["label"] for result in predictions
]
retrieval_df["nli_score"] = [
    result["score"] for result in predictions
]
retrieval_df

,claim,evidence,similarity,nli_label,nli_score
0,The James Webb Space Telescope was launched in...,The James Webb Space Telescope is a space tele...,0.709175,NEUTRAL,0.993388
1,It was launched using an Ariane 5 rocket.,"It was launched on December 25, 2021, aboard a...",0.719603,ENTAILMENT,0.993144
2,The telescope is approximately 1.5 million kil...,The telescope operates near the second Lagrang...,0.809263,ENTAILMENT,0.990595
3,The telescope was launched from India.,The telescope operates near the second Lagrang...,0.519770,NEUTRAL,0.959988


## 11. Final Claim Status

In [11]:
def get_status(label):
    if label == "ENTAILMENT":
        return "Supported"
    elif label == "CONTRADICTION":
        return "Contradicted"
    else:
        return "Not supported by evidence"
retrieval_df["status"] = retrieval_df["nli_label"].apply(get_status)
retrieval_df[[
    "claim",
    "evidence",
    "status",
    "nli_score"
]]

,claim,evidence,status,nli_score
0,The James Webb Space Telescope was launched in...,The James Webb Space Telescope is a space tele...,Not supported by evidence,0.993388
1,It was launched using an Ariane 5 rocket.,"It was launched on December 25, 2021, aboard a...",Supported,0.993144
2,The telescope is approximately 1.5 million kil...,The telescope operates near the second Lagrang...,Supported,0.990595
3,The telescope was launched from India.,The telescope operates near the second Lagrang...,Not supported by evidence,0.959988


## 12. Consistency Score

In [12]:
supported = (
    retrieval_df["nli_label"] == "ENTAILMENT"
).sum()
total = len(retrieval_df)
consistency_score = (supported / total) * 100
print(f"Consistency Score: {consistency_score:.2f}%")

Consistency Score: 50.00%


## 13. Final Results

In [13]:
retrieval_df[[
    "claim",
    "status",
    "nli_score"
]]

,claim,status,nli_score
0,The James Webb Space Telescope was launched in...,Not supported by evidence,0.993388
1,It was launched using an Ariane 5 rocket.,Supported,0.993144
2,The telescope is approximately 1.5 million kil...,Supported,0.990595
3,The telescope was launched from India.,Not supported by evidence,0.959988


## 14. Observations

The complete pipeline connects claim extraction, semantic retrieval, and NLI to check whether generated claims are supported by the source document.